# ***Extracción de de documentos normativos del Ministerio de Salud (MinSalud) mediante API REST***

**Descripción:**  
Este notebook extrae de forma automatizada los documentos normativos (Decretos, Resoluciones, Leyes, etc.) publicados por el Ministerio de Salud y Protección Social de Colombia.

## *Imports y configuración global*

In [2]:
import requests
import pandas as pd
import urllib.parse
import time
import os

# Directorio local donde se guardarán los archivos resultantes.
OUTPUT_DIR = "data"

BASE_URL = "https://www.minsalud.gov.co"
# Endpoint oficial de SharePoint apuntando al identificador (GUID) de la tabla de normatividad.
API_REST = f"{BASE_URL}/_api/web/lists(guid'262f8fee-1318-4201-b8fe-07f9a6994caa')/items"

# Rango de años a consultar. 
YEARS = list(range(2026, 2022, -1))

# Tipos de documentos normativos que nos interesa conservar.
TIPOS_PERMITIDOS =[
    "Concepto", "Decreto", "Ley", "Proyecto de Ley",
    "Proyecto Decreto", "Proyecto Otros", "Proyecto Resolución",
    "Resolución", "Resolución CRES"
]

# Cabeceras HTTP (Headers)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json;odata=verbose"
}

## *Funciones de extracción y procesamiento*

In [ ]:
def scrape_year_rest(session, year):
    """
    Consulta la API REST de SharePoint para obtener todas las normativas
    de la temática "Salud" correspondientes a un año específico.
    
    Args:
        session (requests.Session): Sesión HTTP persistente.
        year (int): El año que se desea consultar.
        
    Returns:
        list[dict]: Lista de diccionarios, donde cada diccionario representa un documento.
    """
    docs =[]

    params = {
        "$filter": f"Tem_x00e1_tica eq 'Salud' and A_x00f1_o eq '{year}'",
        "$select": "FileLeafRef,FileRef,Descripci_x00f3_n,Tipo_x0020_de_x0020_Norma,Tem_x00e1_tica,Subtema,Publicaci_x00f3_n",
        "$top": "5000"
    }

    print(f"  [Año {year}] Descargando datos desde la API REST...")

    try:
        resp = session.get(API_REST, headers=HEADERS, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"    ✗ Error HTTP: {e}")
        if 'resp' in locals(): print(f"    ✗ Detalle: {resp.text[:300]}")
        return docs

    resultados = data.get("d", {}).get("results",[])
    print(f"    → {len(resultados)} normas de Salud encontradas en el servidor.")

    guardados = 0
    for row in resultados:
        tipo_norma = row.get("Tipo_x0020_de_x0020_Norma", "")

        if tipo_norma in TIPOS_PERMITIDOS:
            ruta_archivo = row.get("FileRef", "")
            enlace = urllib.parse.urljoin(BASE_URL, ruta_archivo) if ruta_archivo else ""

            docs.append({
                'nombre':      row.get("FileLeafRef", ""),
                'href':        enlace,
                'descripcion': row.get("Descripci_x00f3_n", ""),
                'tipo_norma':  tipo_norma,
                'tematica':    row.get("Tem_x00e1_tica", ""),
                'subtema':     row.get("Subtema", ""),
                'publicacion': row.get("Publicaci_x00f3_n", "")
            })
            guardados += 1

    print(f"    → {guardados} normas pasaron tu filtro de Tipos de Norma.")
    return docs

def scrape_todos(years=YEARS):
    """
    Itera sobre la lista de años, ejecuta la extracción para cada uno,
    agrupa los resultados y elimina posibles duplicados.
    
    Args:
        years (list[int]): Lista de años a procesar.
        
    Returns:
        pd.DataFrame: DataFrame consolidado con toda la información limpia.
    """
    session = requests.Session()
    todos =[]

    for year in years:
        print(f"\n{'='*50}\nProcesando año {year}...")
        docs = scrape_year_rest(session, year)
        todos.extend(docs)
        time.sleep(1.5) 

    df = pd.DataFrame(todos)
    if not df.empty:
        df = df.drop_duplicates(subset='href')
        print(f"Total de documentos filtrados guardados: {len(df)}")
    return df

## *Ejecución del scraping y exportación de archivos*

In [4]:
df = scrape_todos(years=YEARS)

if df.empty:
    print("\n⚠️ ERROR: El DataFrame sigue vacío.")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    ruta_csv = os.path.join(OUTPUT_DIR, 'minsalud_normatividad_filtrado.csv')
    ruta_txt = os.path.join(OUTPUT_DIR, 'enlaces_minsalud.txt')

    df.to_csv(ruta_csv, index=False, encoding='utf-8-sig')
    print(f"✅ Archivo CSV guardado en: {ruta_csv}")
    
    with open(ruta_txt, 'w', encoding='utf-8') as f:
        for link in df['href']:
            if link: 
                f.write(f"{link}\n")
                
    print(f"✅ Archivo de enlaces TXT guardado en: {ruta_txt}")

    print("\nPrimeros resultados guardados:")
    print(df[['nombre', 'tipo_norma', 'publicacion']].head(10))


Procesando año 2026...
  [Año 2026] Descargando datos desde la API REST...
    → 52 normas de Salud encontradas en el servidor.
    → 38 normas pasaron tu filtro de Tipos de Norma.

Procesando año 2025...
  [Año 2025] Descargando datos desde la API REST...
    → 246 normas de Salud encontradas en el servidor.
    → 185 normas pasaron tu filtro de Tipos de Norma.

Procesando año 2024...
  [Año 2024] Descargando datos desde la API REST...
    → 221 normas de Salud encontradas en el servidor.
    → 180 normas pasaron tu filtro de Tipos de Norma.

Procesando año 2023...
  [Año 2023] Descargando datos desde la API REST...
    → 209 normas de Salud encontradas en el servidor.
    → 155 normas pasaron tu filtro de Tipos de Norma.

✅ ¡ÉXITO TOTAL! Total de documentos filtrados guardados: 558
✅ Archivo CSV guardado en: data\minsalud_normatividad_filtrado.csv
✅ Archivo de enlaces TXT guardado en: data\enlaces_minsalud.txt

Primeros resultados guardados:
                          nombre  tipo_no